In [1]:
import os

os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["hadoop.home.dir"] = r"C:\hadoop"
os.environ["PATH"] += r";C:\hadoop\bin"

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, to_timestamp, window, sum, count, when
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("AggregationNotebook") \
    .master("local[*]") \
    .config(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0"
    ) \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

print("Spark session started")

Spark session started


In [2]:
order_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("price", DoubleType(), True),
    StructField("event_time", StringType(), True)
])

print("Schema created")

Schema created


In [3]:
df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "127.0.0.1:9092") \
    .option("subscribe", "order_events") \
    .option("startingOffsets", "latest") \
    .option("failOnDataLoss", "false") \
    .load()

print("Kafka stream connected")

Kafka stream connected


In [4]:
dedup_df = df.selectExpr(
    "CAST(value AS STRING) as json_data"
).select(
    from_json(col("json_data"), order_schema).alias("order_data")
).select(
    col("order_data.order_id").alias("order_id"),
    col("order_data.customer_id").alias("customer_id"),
    col("order_data.product_id").alias("product_id"),
    col("order_data.event_type").alias("event_type"),
    col("order_data.quantity").alias("quantity"),
    col("order_data.price").alias("price"),
    to_timestamp(col("order_data.event_time")).alias("event_timestamp")
).withWatermark(
    "event_timestamp", "10 minutes"
).dropDuplicates([
    "order_id",
    "product_id",
    "event_type",
    "event_timestamp"
])

print("Deduplicated event-time dataframe ready")

Deduplicated event-time dataframe ready


In [5]:
order_value_window_df = dedup_df.groupBy(
    window(col("event_timestamp"), "5 minutes"),
    col("customer_id")
).agg(
    sum(col("quantity") * col("price")).alias("total_order_value")
)

print("Order value window aggregation created")

Order value window aggregation created


In [6]:


#Count cancelled orders per 5-minute window

cancelled_orders_window_df = dedup_df.groupBy(
    window(col("event_timestamp"), "5 minutes")
).agg(
    count(
        when(col("event_type") == "CANCELLED", 1)
    ).alias("cancelled_orders_count")
)

print("Cancelled orders window aggregation created")

Cancelled orders window aggregation created


In [7]:
order_value_query = order_value_window_df.writeStream \
    .format("memory") \
    .queryName("order_value_window_check") \
    .outputMode("update") \
    .start()

print("Order value validation stream started")

Order value validation stream started


In [8]:
cancelled_query = cancelled_orders_window_df.writeStream \
    .format("memory") \
    .queryName("cancelled_orders_check") \
    .outputMode("complete") \
    .start()

print("Cancelled count validation stream started")

Cancelled count validation stream started


In [9]:
order_value_query.awaitTermination(10)


False

In [10]:
cancelled_query.awaitTermination(10)

False

In [11]:
spark.sql("""
SELECT *
FROM order_value_window_check
ORDER BY window.start DESC
LIMIT 20
""").show(truncate=False)

+------------------------------------------+-----------+------------------+
|window                                    |customer_id|total_order_value |
+------------------------------------------+-----------+------------------+
|{2026-03-31 13:00:00, 2026-03-31 13:05:00}|cust_15    |65.93             |
|{2026-03-31 13:00:00, 2026-03-31 13:05:00}|cust_17    |560.04            |
|{2026-03-31 13:00:00, 2026-03-31 13:05:00}|cust_15    |224.93            |
|{2026-03-31 13:00:00, 2026-03-31 13:05:00}|cust_3     |183.35999999999999|
|{2026-03-31 13:00:00, 2026-03-31 13:05:00}|cust_15    |532.05            |
|{2026-03-31 13:00:00, 2026-03-31 13:05:00}|cust_4     |718.4499999999999 |
|{2026-03-31 13:00:00, 2026-03-31 13:05:00}|cust_6     |922.4             |
|{2026-03-31 13:00:00, 2026-03-31 13:05:00}|cust_16    |1495.1899999999998|
|{2026-03-31 13:00:00, 2026-03-31 13:05:00}|cust_13    |163.15            |
|{2026-03-31 13:00:00, 2026-03-31 13:05:00}|cust_4     |682.52            |
|{2026-03-31

In [12]:
spark.sql("""
SELECT *
FROM cancelled_orders_check
ORDER BY window.start DESC
LIMIT 20
""").show(truncate=False)

+------------------------------------------+----------------------+
|window                                    |cancelled_orders_count|
+------------------------------------------+----------------------+
|{2026-03-31 13:00:00, 2026-03-31 13:05:00}|1                     |
|{2026-03-31 12:55:00, 2026-03-31 13:00:00}|6                     |
+------------------------------------------+----------------------+

